In [0]:
spark.sql("""create database if not exists freshcart_team1""")


In [0]:
from pyspark.sql import functions as f
import uuid 
    

In [0]:

SCHEMA = "freshcart_team1"
files = [
    "customers",
    "deliveries",
    "order_items",
    "orders",
    "products"
]
BASE_PATH = f"/Volumes/de_workspace26/{SCHEMA}"


run_id = str(uuid.uuid4())

df_dict={}

# first we read all the raw daya from volume and add some MetaData Coloumns on it 

for file in files:
    file_path = f"{BASE_PATH}/{file}"
    print(f"processing:{file_path}")
    
    df=(spark.read.csv(file_path,header=True,inferSchema=True))

    df=(
        
          df.withColumn("_source", f.lit("volume"))\
          .withColumn("_ingest_ts", f.current_timestamp())\
           .withColumn("_file_name", f.lit(f"{file}"))
          .withColumn("_run_id", f.lit(run_id))\
          .withColumn("ingest_date", f.to_date(f.current_timestamp()))\
    )
    df.show(5)

    df_dict[file]=df


   

In [0]:

# now we write all the deta in delta form in same catalog_schema_table 

for file, df in df_dict.items():
    
    table_name = f"freshcart_team1.bronze_{file}"
    
    (df.write
        .format("delta")
        .mode("append")   
        .partitionBy("ingest_date")
        .option("mergeSchema", "true")
        .saveAsTable(table_name)
    )
    spark.sql(f"""
        ALTER TABLE {table_name}
        SET TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true'
        )
    """)

    

In [0]:
%sql



DELETE FROM  de_workspace26.freshcart_team1.bronze_orders
WHERE order_id IS NULL OR total_amount < 0;





In [0]:
%sql


alter table de_workspace26.freshcart_team1.bronze_orders
add constraint validate_order_id check(order_id IS NOT NULL);

--  this will throw error because total_amount is string type will fix these things in silver so wait till silver
alter table de_workspace26.freshcart_team1.bronze_orders
add constraint valid_payments check(total_amount>=0)


In [0]:
%sql

show tables in freshcart_team1;

In [0]:
%sql
select count(*) from de_workspace26.freshcart_team1.bronze_orders;

select count(*) from de_workspace26.freshcart_team1.bronze_deliveries;

select count(*) from  de_workspace26.freshcart_team1.bronze_order_items;


select count(*) from  de_workspace26.freshcart_team1.bronze_customers;

select count(*) from de_workspace26.freshcart_team1.bronze_products;


In [0]:
df = spark.read.table("de_workspace26.freshcart_team1.bronze_customers")

df.show()




In [0]:
df=df.withColumn('random_data',f.rand())

df.write.format("delta").mode("append").saveAsTable("de_workspace26.freshcart_team1.bronze_customers")

In [0]:
df_bronze_order=spark.read.format('delta').table('freshcart_team1.bronze_orders')

df_bronze_order.show(5)

In [0]:
df_bronze_order.filter(f.col('status')=='DELIVERED').select('order_id','customer_id','total_amount').explain(True)